# Symbolic Operator Calculus — derivación semántica K7C

Esta libreta consume las APIs públicas del paquete; no reconstruye la derivación matemática.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import sympy as sp
from IPython.display import Math, display
import symbolic_operator_calculus as soc


## 1. Traza real y renderizado semántico


In [ ]:
x, y, u, v = sp.symbols('x y u v', positive=True)
f = sp.Function('f')
r11_out, r11_in = sp.symbols('r11_out r11_in', positive=True)
regularizer_kernel = soc.KernelRepresentation(
    operator=soc.a11_formal_regularizer(),
    kernel_expression=sp.Function('R11')(r11_out, r11_in),
    output_variable=r11_out,
    input_variable=r11_in,
    integration_domain=soc.IntegrationDomain(0, sp.oo, 'positive half-line'),
    semantic_status=soc.KernelRepresentationStatus.FORMAL,
    hypotheses=('explicit formal kernel assumption for this notebook',),
)
rules = soc.mvp_atomic_rules(regularizer_kernel=regularizer_kernel)
trace = soc.build_first_schur_derivation_trace(
    f(x),
    x,
    input_variable=y,
    outer_variable=u,
    middle_variable=v,
    rules=rules,
    regularizer_kernel=regularizer_kernel,
)
rendered = soc.render_first_schur_derivation_latex(trace)
rendered_by_key = {step.key: step for step in rendered.steps}

for step in rendered.steps:
    print(step.title)
    display(Math(step.latex))


## 2. Corrección expandida aplicada a una función

El coeficiente exterior y la expresión aplicada se muestran por separado.


In [ ]:
correction = soc.a22_first_schur_correction()
applied = soc.apply_linear_combination_ordered(
    correction,
    f(x),
    x,
    rules,
)

for index, term in enumerate(applied.terms, start=1):
    print(f'Término {index}; coeficiente = {term.coefficient}')
    display(Math(soc.render_scalar_latex(term.expression)))


## 3. Kernel combinado y su acción


In [ ]:
combined_kernel = soc.combined_kernel_c22(
    x,
    y,
    outer_variable=u,
    middle_variable=v,
    rules=rules,
    regularizer_kernel=regularizer_kernel,
)
assert combined_kernel == trace.combined_kernel
display(Math(rendered_by_key['combined_kernel'].latex))

kernel_action = soc.apply_combined_kernel_c22(
    x,
    f,
    y,
    outer_variable=u,
    middle_variable=v,
    rules=rules,
    regularizer_kernel=regularizer_kernel,
)
display(Math(soc.render_combined_kernel_action_latex(kernel_action, x, f, y)))


## 4. Modelos de los bloques fuera de la diagonal


In [ ]:
relation21 = soc.a21_mod_compact_relation()
relation12 = soc.a12_mod_compact_relation()
assert trace.offdiagonal_relations == (relation21, relation12)
display(Math(rendered_by_key['offdiagonal_models'].latex))


## 5. Kernels laterales en orden semántico


In [ ]:
assert trace.m21_combination == soc.m21_kernel_combination(x, u, rules=rules)
assert trace.m12_combination == soc.m12_kernel_combination(v, y, rules=rules)
display(Math(rendered_by_key['left_kernel'].latex))
display(Math(rendered_by_key['right_kernel'].latex))


## 6. Realización Fourier explícita breve

Las acciones de Wiener–Hopf se realizan con los kernels racionales normalizados; el regularizador permanece formal.


In [ ]:
d = soc.positive_decay_symbol()
explicit_rules = soc.explicit_wiener_hopf_rules(decay=d)
explicit_trace = soc.build_first_schur_derivation_trace(
    f(x),
    x,
    input_variable=y,
    outer_variable=u,
    middle_variable=v,
    rules=explicit_rules,
    regularizer_kernel=regularizer_kernel,
)
explicit_rendered = soc.render_first_schur_derivation_latex(explicit_trace)
explicit_by_key = {step.key: step for step in explicit_rendered.steps}
display(Math(explicit_by_key['left_kernel'].latex))
display(Math(explicit_by_key['right_kernel'].latex))


## 7. Exportación del resultado ya renderizado

La demostración escribe únicamente en un directorio temporal y no genera PDF.


In [ ]:
with TemporaryDirectory() as temporary_directory:
    destination = Path(temporary_directory) / 'first-schur.tex'
    soc.export_first_schur_derivation_tex(rendered, destination)
    exported_document = destination.read_text(encoding='utf-8')

print(exported_document)
